In [1]:
!nvidia-smi
!pip -q install -U kaggle

import os, time, random, gc, zipfile, shutil, subprocess
from pathlib import Path, PurePosixPath
from collections import defaultdict
import numpy as np, pandas as pd
from PIL import Image, ImageOps, ImageFile
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.model_selection import train_test_split

assert torch.cuda.is_available(), "Aktifkan GPU: Runtime > Change runtime type > GPU."
DEVICE=torch.device("cuda")
print("GPU:",torch.cuda.get_device_name(0))
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark=True
IMG_SIZE=260
BATCH_SIZE=32 if torch.cuda.get_device_properties(0).total_memory >= 10*1024**3 else 16
EPOCHS=5
LR=2e-4
NUM_WORKERS=2
print("Batch size:",BATCH_SIZE)

Wed Nov  4 14:21:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Kaggle API authentication

In [2]:
from google.colab import files
uploaded=files.upload()  # Upload kaggle.json from Kaggle Settings > API
if "kaggle.json" not in uploaded: raise FileNotFoundError("Upload kaggle.json.")
os.makedirs("/root/.kaggle",exist_ok=True)
shutil.copy("kaggle.json","/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json",0o600)
print("Kaggle API configured.")

Saving kaggle.json to kaggle.json
Kaggle API configured.


## 2. Download competition dataset (accept competition rules on Kaggle first)

In [11]:
COMP="document-image-classification-sd234504"
ROOT=Path("/content/document_data"); ROOT.mkdir(exist_ok=True,parents=True)
r=subprocess.run(["kaggle","competitions","download","-c",COMP,"-p",str(ROOT)],capture_output=True,text=True)
print(r.stdout)
if r.returncode: print(r.stderr); raise RuntimeError("Download gagal. Cek kaggle.json dan pastikan sudah Join/Accept Rules.")
zips=list(ROOT.glob("*.zip"))
if not zips: raise FileNotFoundError("ZIP kompetisi tidak ditemukan.")
EXTRACT=ROOT/"data"; EXTRACT.mkdir(exist_ok=True)
for z in zips:
    with zipfile.ZipFile(z) as f: f.extractall(EXTRACT)
print("Extracted:",EXTRACT)

document-image-classification-sd234504.zip: Skipping, found more recently modified local copy (use --force to force download)

Extracted: /content/document_data/data


## 3. CSV and image path discovery

In [12]:
def unique_file(root,name):
    hits=list(root.rglob(name))
    if len(hits)!=1: raise FileNotFoundError(f"Expected one {name}, found {len(hits)}")
    return hits[0]
train_df=pd.read_csv(unique_file(EXTRACT,"train.csv"))
test_df=pd.read_csv(unique_file(EXTRACT,"test.csv"))
sample_sub=pd.read_csv(unique_file(EXTRACT,"sample_submission.csv"))
print(train_df.shape,test_df.shape,sample_sub.shape)
EXTS={".tif",".tiff",".png",".jpg",".jpeg"}
paths=[p for p in EXTRACT.rglob("*") if p.is_file() and p.suffix.lower() in EXTS]
idx=defaultdict(list)
for p in paths: idx[p.name].append(p)
def resolve(name):
    parts=PurePosixPath(str(name).replace("\\\\","/").lstrip("/")).parts
    if not parts: return None
    candidates=idx.get(parts[-1],[])
    for n in range(len(parts),0,-1):
        suffix=tuple(parts[-n:])
        hits=[p for p in candidates if tuple(p.parts[-n:])==suffix]
        if len(hits)==1: return str(hits[0])
        if len(hits)>1: raise RuntimeError(f"Ambiguous path: {name}")
    return str(candidates[0]) if len(candidates)==1 else None
train_df["image_path"]=train_df.filename.map(resolve)
test_df["image_path"]=test_df.filename.map(resolve)
print("Images found:",len(paths))
print("Train resolved:",train_df.image_path.notna().sum(),"/",len(train_df))
print("Test resolved:",test_df.image_path.notna().sum(),"/",len(test_df))
if train_df.image_path.isna().any():
    display(train_df[train_df.image_path.isna()][["id","filename"]].head(20)); raise FileNotFoundError("Train paths unresolved.")
if test_df.image_path.isna().any():
    display(test_df[test_df.image_path.isna()][["id","filename"]].head(20)); raise FileNotFoundError("Test paths unresolved.")

(32003, 4) (7994, 2) (7994, 2)
Images found: 39997
Train resolved: 32003 / 32003
Test resolved: 7994 / 7994


## 4. Validate images (test rows are never dropped)

In [5]:
ImageFile.LOAD_TRUNCATED_IMAGES=False
def valid_image(p):
    try:
        with Image.open(p) as im: im.load()
        return True
    except Exception: return False
train_df["valid"]=[valid_image(p) for p in train_df.image_path]
test_df["valid"]=[valid_image(p) for p in test_df.image_path]
print("Train readable:",train_df.valid.sum(),"/",len(train_df))
print("Test readable:",test_df.valid.sum(),"/",len(test_df))
if (~test_df.valid).any():
    display(test_df.loc[~test_df.valid,["id","filename","image_path"]].head(20))
    raise RuntimeError("Test image unreadable; do not drop test rows.")
train_df=train_df[train_df.valid].copy().reset_index(drop=True)
if train_df.empty: raise RuntimeError("No valid train images.")

/usr/local/lib/python3.13/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Corrupt EXIF data.  Expecting to read 2 bytes but only got 0. 
  warnings.warn(str(msg))


Train readable: 32002 / 32003
Test readable: 7994 / 7994


## 5. Labels and stratified split

In [6]:
train_df["label"]=train_df.label.astype(int)
classes=sorted(train_df.label.unique().tolist())
N_CLASSES=len(classes)
if classes!=list(range(N_CLASSES)): raise ValueError(f"Unexpected labels: {classes}")
class_names=(train_df[["label","class_name"]].drop_duplicates().sort_values("label").class_name.tolist()
             if "class_name" in train_df else [str(x) for x in classes])
print("Classes:",N_CLASSES,class_names)
display(train_df.label.value_counts().sort_index().to_frame("count"))
tr,va=train_test_split(train_df,test_size=0.12,random_state=SEED,stratify=train_df.label)
tr=tr.reset_index(drop=True); va=va.reset_index(drop=True)
print("Train:",len(tr),"Validation:",len(va))

Classes: 16 ['advertisement', 'budget', 'email', 'file_folder', 'form', 'handwritten', 'invoice', 'letter', 'memo', 'news_article', 'presentation', 'questionnaire', 'resume', 'scientific_publication', 'scientific_report', 'specification']


,count
label,
0,2012
1,2004
2,2013
3,2022
4,2005
5,2026
6,1982
7,1972
8,1994


Train: 28161 Validation: 3841


## 6. Preprocessing and DataLoaders

In [7]:
class ResizePad:
    def __init__(self,size): self.size=size
    def __call__(self,im):
        im=ImageOps.exif_transpose(im).convert("RGB")
        im.thumbnail((self.size,self.size),Image.Resampling.LANCZOS)
        canvas=Image.new("RGB",(self.size,self.size),(255,255,255))
        canvas.paste(im,((self.size-im.width)//2,(self.size-im.height)//2))
        return canvas
mean=[0.485,0.456,0.406]; std=[0.229,0.224,0.225]
train_tf=transforms.Compose([ResizePad(IMG_SIZE),transforms.RandomRotation(2,fill=(255,255,255)),
    transforms.ColorJitter(brightness=.06,contrast=.06),transforms.ToTensor(),transforms.Normalize(mean,std)])
eval_tf=transforms.Compose([ResizePad(IMG_SIZE),transforms.ToTensor(),transforms.Normalize(mean,std)])
class DocDataset(Dataset):
    def __init__(self,df,tfm,test=False): self.df=df.reset_index(drop=True); self.tfm=tfm; self.test=test
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        r=self.df.iloc[i]
        with Image.open(r.image_path) as im: im=ImageOps.exif_transpose(im).convert("RGB")
        x=self.tfm(im)
        return (x,int(r.id)) if self.test else (x,int(r.label))
train_loader=DataLoader(DocDataset(tr,train_tf),batch_size=BATCH_SIZE,shuffle=True,num_workers=NUM_WORKERS,pin_memory=True,persistent_workers=True)
valid_loader=DataLoader(DocDataset(va,eval_tf),batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,persistent_workers=True)
test_loader=DataLoader(DocDataset(test_df,eval_tf,True),batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,persistent_workers=True)
print("Batch check:",next(iter(train_loader))[0].shape)

Batch check: torch.Size([32, 3, 260, 260])


## 7. Pretrained EfficientNet-B2

In [8]:
# Downloads official ImageNet weights into Colab. Requires Colab internet.
weights=models.EfficientNet_B2_Weights.DEFAULT
model=models.efficientnet_b2(weights=weights)
in_features=model.classifier[1].in_features
model.classifier[1]=nn.Sequential(nn.Dropout(.30),nn.Linear(in_features,N_CLASSES))
model=model.to(DEVICE)
criterion=nn.CrossEntropyLoss(label_smoothing=.05)
optimizer=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=1e-4)
scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=EPOCHS,eta_min=LR*.08)
scaler=torch.amp.GradScaler("cuda")
print("Pretrained EfficientNet-B2 ready.")

Downloading: "https://download.pytorch.org/models/efficientnet_b2_rwightman-c35c1473.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b2_rwightman-c35c1473.pth


100%|██████████| 35.2M/35.2M [00:00<00:00, 92.6MB/s]


Pretrained EfficientNet-B2 ready.


## 8. Train for up to 5 epochs (GPU AMP + early stopping)

In [9]:
@torch.no_grad()
def evaluate(loader):
    model.eval(); loss_sum=correct=count=0
    for x,y in loader:
        x=x.to(DEVICE,non_blocking=True); y=y.to(DEVICE,non_blocking=True)
        with torch.autocast("cuda",dtype=torch.float16):
            logits=model(x); loss=criterion(logits,y)
        n=y.size(0); loss_sum+=loss.item()*n; correct+=(logits.argmax(1)==y).sum().item(); count+=n
    return loss_sum/count,correct/count

BEST="/content/best_efficientnet_b2.pth"
best_acc=0.; stale=0; patience=2; logs=[]
t_all=time.time()
for epoch in range(1,EPOCHS+1):
    t0=time.time(); model.train(); loss_sum=correct=count=0
    for step,(x,y) in enumerate(train_loader,1):
        x=x.to(DEVICE,non_blocking=True); y=y.to(DEVICE,non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast("cuda",dtype=torch.float16):
            logits=model(x); loss=criterion(logits,y)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(model.parameters(),2.)
        scaler.step(optimizer); scaler.update()
        n=y.size(0); loss_sum+=loss.item()*n; correct+=(logits.argmax(1)==y).sum().item(); count+=n
        if step%100==0: print(f"epoch {epoch} batch {step}/{len(train_loader)}",flush=True)
    tr_loss=loss_sum/count; tr_acc=correct/count
    va_loss,va_acc=evaluate(valid_loader); scheduler.step()
    mins=(time.time()-t0)/60
    logs.append([epoch,tr_loss,tr_acc,va_loss,va_acc,mins])
    print(f"Epoch {epoch}/{EPOCHS} | train acc {tr_acc:.4%} | val acc {va_acc:.4%} | val loss {va_loss:.4f} | {mins:.1f} min",flush=True)
    if va_acc>best_acc:
        best_acc=va_acc; stale=0; torch.save(model.state_dict(),BEST); print("Best checkpoint saved.")
    else: stale+=1
    if stale>=patience:
        print("Early stopping."); break
    gc.collect(); torch.cuda.empty_cache()
print("Total minutes:",round((time.time()-t_all)/60,1),"Best val accuracy:",round(best_acc*100,3))
history_df=pd.DataFrame(logs,columns=["epoch","train_loss","train_acc","val_loss","val_acc","minutes"])
display(history_df)

epoch 1 batch 100/881
epoch 1 batch 200/881
epoch 1 batch 300/881
epoch 1 batch 400/881
epoch 1 batch 500/881
epoch 1 batch 600/881
epoch 1 batch 700/881
epoch 1 batch 800/881
Epoch 1/5 | train acc 66.1731% | val acc 78.2348% | val loss 0.9583 | 12.0 min
Best checkpoint saved.
epoch 2 batch 100/881
epoch 2 batch 200/881
epoch 2 batch 300/881
epoch 2 batch 400/881
epoch 2 batch 500/881
epoch 2 batch 600/881
epoch 2 batch 700/881
epoch 2 batch 800/881
Epoch 2/5 | train acc 80.4979% | val acc 82.3483% | val loss 0.8406 | 11.1 min
Best checkpoint saved.
epoch 3 batch 100/881
epoch 3 batch 200/881
epoch 3 batch 300/881
epoch 3 batch 400/881
epoch 3 batch 500/881
epoch 3 batch 600/881
epoch 3 batch 700/881
epoch 3 batch 800/881
Epoch 3/5 | train acc 85.7995% | val acc 84.0927% | val loss 0.7887 | 10.8 min
Best checkpoint saved.
epoch 4 batch 100/881
epoch 4 batch 200/881
epoch 4 batch 300/881
epoch 4 batch 400/881
epoch 4 batch 500/881
epoch 4 batch 600/881
epoch 4 batch 700/881
epoch 4 batc

,epoch,train_loss,train_acc,val_loss,val_acc,minutes
0,1,1.312233,0.661731,0.958305,0.782348,11.989914
1,2,0.897170,0.804979,0.840579,0.823483,11.082463
2,3,0.741493,0.857995,0.788669,0.840927,10.841102
3,4,0.617275,0.901992,0.769673,0.854465,10.813748
4,5,0.539451,0.929228,0.770317,0.855767,10.838634


In [13]:
class DocDataset(Dataset):
    def __init__(self, df, tfm, test=False):
        self.df = df.reset_index(drop=True)
        self.tfm = tfm
        self.test = test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        r = self.df.iloc[i]

        with Image.open(r.image_path) as im:
            im = ImageOps.exif_transpose(im).convert("RGB")

        x = self.tfm(im)

        if self.test:
            # Pertahankan ID sebagai string, jangan konversi ke integer
            return x, str(r.id)

        return x, int(r.label)

In [14]:
test_loader = DataLoader(
    DocDataset(test_df, eval_tf, test=True),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True
)

In [15]:
model.load_state_dict(torch.load(BEST, map_location=DEVICE))
model.eval()

ids = []
preds = []

with torch.no_grad():
    for x, batch_ids in test_loader:

        x = x.to(DEVICE, non_blocking=True)

        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(x)

        predictions = logits.argmax(1).cpu().numpy()

        preds.extend(predictions.tolist())
        ids.extend(list(batch_ids))

pred_df = pd.DataFrame({
    "id": ids,
    "label": preds
})

print("Total predictions:", len(pred_df))
display(pred_df.head())

Total predictions: 7994


,id,label
0,test_00001,0
1,test_00002,0
2,test_00003,1
3,test_00004,4
4,test_00005,13


## 9. Predict test and create submission

In [16]:
submission = sample_sub[["id"]].merge(
    pred_df,
    on="id",
    how="left",
    validate="one_to_one"
)

assert submission["label"].notna().all()
assert len(submission) == len(sample_sub)
assert submission["id"].equals(sample_sub["id"])

submission["label"] = submission["label"].astype(int)

submission.to_csv("/content/submission.csv", index=False)

print("Submission berhasil dibuat!")
print("Shape:", submission.shape)
display(submission.head())

from google.colab import files
files.download("/content/submission.csv")

Submission berhasil dibuat!
Shape: (7994, 2)


,id,label
0,test_00001,0
1,test_00002,0
2,test_00003,1
3,test_00004,4
4,test_00005,13


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>